# Boundary Evaluation on Colab

This notebook runs the baseline and hard event-boundary datasets with timestamped outputs.


In [ ]:
%cd /content
!ls

## Clone or Update Repository

Use a GitHub token if the repository is private. If the repository already exists, pull the latest version instead of cloning again.


In [ ]:
from getpass import getpass
import os

REPO_URL = "github.com/norewyx0205/vlm-event-boundary.git"
REPO_DIR = "/content/vlm-event-boundary"

if not os.path.exists(REPO_DIR):
    token = getpass("GitHub token (leave empty for public repo): ")
    if token:
        !git clone https://{token}@{REPO_URL} {REPO_DIR}
    else:
        !git clone https://{REPO_URL} {REPO_DIR}
else:
    print("Repository already exists; pulling latest changes...")
    %cd {REPO_DIR}
    !git pull


In [ ]:
%cd /content/vlm-event-boundary
!ls

## Install Dependencies

`qwen-vl-utils` and `decord` are required for video input processing. Restart the runtime if Colab asks after installing/upgrading packages.


In [ ]:
!pip install -U transformers accelerate qwen-vl-utils decord

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    torch.set_default_device("cuda")


## Check Dataset Files

The baseline set is the easy sanity-check dataset. The hard set is the longer counterbalanced dataset with distractors.


In [ ]:
from pathlib import Path

for dataset_dir in ["baseline_boundary_videos", "synthetic_boundary_videos"]:
    ann = Path(dataset_dir) / "annotations.jsonl"
    video_dir = Path(dataset_dir) / "videos"
    print(dataset_dir)
    print("  annotation exists:", ann.exists())
    print("  eval rows:", sum(1 for _ in open(ann)) if ann.exists() else 0)
    print("  videos:", len(list(video_dir.glob("*.mp4"))) if video_dir.exists() else 0)


## Optional: Regenerate Datasets

Only run this cell if you want to regenerate both datasets from the current generator.


In [ ]:
# !python generate_2d_boundary_videos.py --dataset all

## Evaluation Configuration

Change `MODEL_NAME` to compare Qwen2-VL and stronger Qwen3-VL models. Results are saved under `results/<experiment-version>/<model>/<timestamp>/`.


In [ ]:
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"
# MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"

RESULT_DIR = "/content/vlm-event-boundary/results"
BASELINE_ANNOTATION = "/content/vlm-event-boundary/baseline_boundary_videos/annotations.jsonl"
HARD_ANNOTATION = "/content/vlm-event-boundary/synthetic_boundary_videos/annotations.jsonl"


## Smoke Test

Run a tiny subset first to confirm the model and video pipeline work.


In [ ]:
!python run_eval.py \
  --model-name "$MODEL_NAME" \
  --annotation-path "$BASELINE_ANNOTATION" \
  --experiment-version baseline_smoke_test \
  --result-dir "$RESULT_DIR" \
  --max-samples 4


## Run Baseline Evaluation

This should remain relatively easy for the model and acts as a sanity check.


In [ ]:
!python run_eval.py \
  --model-name "$MODEL_NAME" \
  --annotation-path "$BASELINE_ANNOTATION" \
  --experiment-version baseline \
  --result-dir "$RESULT_DIR"


## Run Hard Evaluation

Use a version label such as `hard_v1`, `hard_v2`, or `hard_v2_prompt_swap` whenever the dataset or prompt design changes.


In [ ]:
!python run_eval.py \
  --model-name "$MODEL_NAME" \
  --annotation-path "$HARD_ANNOTATION" \
  --experiment-version hard_v1 \
  --result-dir "$RESULT_DIR"


## Inspect Saved Results

Each run writes `raw_results.jsonl`, `summary.json`, `summary.txt`, and `config.json` into a timestamped folder.


In [ ]:
!find "$RESULT_DIR" -maxdepth 4 -type f | sort | tail -40